In [17]:
import os
import re
import gc
import sqlite3
import unicodedata
import random
import math

import numpy as np
import pandas as pd

from difflib import SequenceMatcher
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

DATA_DIR = "/home/sagemaker-user/Igniters_submission_file"
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

DB_PATH = os.path.join(DATA_DIR, "entity_index.db")

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Pipeline environment ready.")
print("Database:", DB_PATH)

Pipeline environment ready.
Database: /home/sagemaker-user/Igniters_submission_file/entity_index.db


In [19]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value).lower()

    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")

    text = text.replace("&", " and ")

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_series(series):
    return series.fillna("").astype(str).map(normalize_text)


def first_token(text):
    text = normalize_text(text)

    if not text:
        return ""

    return text.split()[0]


def name_prefix(text, n=4):
    text = normalize_text(text)

    if not text:
        return ""

    return text[:n]


def address_tokens(text):
    text = normalize_text(text)

    if not text:
        return []

    return [
        token for token in text.split()
        if len(token) >= 5
    ]


print("Normalization functions ready.")

Normalization functions ready.


In [20]:
if os.path.exists(DB_PATH):
    print("Existing database found:")
    print(DB_PATH)
    print("If this is from an incomplete previous run, delete it before continuing.")
else:
    print("No existing database. Ready to build.")

Existing database found:
/home/sagemaker-user/Igniters_submission_file/entity_index.db
If this is from an incomplete previous run, delete it before continuing.


In [21]:
conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS entities (
    entity_id TEXT PRIMARY KEY,
    source TEXT NOT NULL,
    business_name TEXT,
    business_address TEXT,
    country TEXT,
    clean_name TEXT,
    clean_address TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS name_prefix_index (
    block_key TEXT,
    entity_id TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS first_token_index (
    block_key TEXT,
    entity_id TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS address_token_index (
    block_key TEXT,
    entity_id TEXT
)
""")

conn.commit()

print("SQLite schema created.")

SQLite schema created.


In [12]:
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_name_prefix
ON name_prefix_index(block_key)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_first_token
ON first_token_index(block_key)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_address_token
ON address_token_index(block_key)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_entities_id
ON entities(entity_id)
""")

conn.commit()

print("SQLite indexes created.")

SQLite indexes created.


In [10]:
# Cell: Memory-safe chunk processor

CHUNK_SIZE = 25_000   # smaller chunk = lower RAM usage


def process_source_file(filepath, source_name):
    print(f"\nProcessing {source_name}")
    print(f"File: {filepath}")

    total = 0

    for chunk in pd.read_csv(
        filepath,
        sep="\t",
        dtype="string",
        chunksize=CHUNK_SIZE
    ):
        # Fill missing values
        chunk = chunk.fillna("")

        # Normalize name and address
        chunk["clean_name"] = normalize_series(chunk["business_name"])
        chunk["clean_address"] = normalize_series(chunk["business_address"])

        # ---------------------------------------------------------
        # 1. Insert main entity records
        # ---------------------------------------------------------

        records = []

        for row in chunk.itertuples(index=False):

            records.append((
                str(row.entity_id),
                source_name,
                str(row.business_name),
                str(row.business_address),
                str(row.country),
                str(row.clean_name),
                str(row.clean_address)
            ))

        conn.executemany(
            """
            INSERT OR REPLACE INTO entities
            (
                entity_id,
                source,
                business_name,
                business_address,
                country,
                clean_name,
                clean_address
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            records
        )

        # ---------------------------------------------------------
        # 2. Create blocking records
        # ---------------------------------------------------------

        prefix_records = []
        first_records = []
        address_records = []

        for row in chunk.itertuples(index=False):

            entity_id = str(row.entity_id)
            country = normalize_text(row.country)
            name = str(row.clean_name)
            address = str(row.clean_address)

            # -----------------------------------------------------
            # Name prefix block
            # -----------------------------------------------------

            if country and name:

                prefix = name[:4]

                if prefix:
                    prefix_records.append(
                        (country + "|" + prefix, entity_id)
                    )

                # -------------------------------------------------
                # First word block
                # -------------------------------------------------

                name_parts = name.split()

                if name_parts:

                    first = name_parts[0]

                    if len(first) >= 3:
                        first_records.append(
                            (country + "|" + first, entity_id)
                        )

            # -----------------------------------------------------
            # Address token block
            # -----------------------------------------------------

            if country and address:

                tokens = address.split()

                used = set()

                for token in tokens:

                    # Ignore very short tokens
                    if len(token) < 5:
                        continue

                    # Ignore pure numbers
                    if token.isdigit():
                        continue

                    # Avoid duplicate tokens
                    if token in used:
                        continue

                    used.add(token)

                    address_records.append(
                        (country + "|" + token, entity_id)
                    )

                    # Maximum 3 address tokens per business
                    if len(used) >= 3:
                        break

        # ---------------------------------------------------------
        # 3. Insert blocking indexes
        # ---------------------------------------------------------

        if prefix_records:

            conn.executemany(
                """
                INSERT INTO name_prefix_index
                (block_key, entity_id)
                VALUES (?, ?)
                """,
                prefix_records
            )

        if first_records:

            conn.executemany(
                """
                INSERT INTO first_token_index
                (block_key, entity_id)
                VALUES (?, ?)
                """,
                first_records
            )

        if address_records:

            conn.executemany(
                """
                INSERT INTO address_token_index
                (block_key, entity_id)
                VALUES (?, ?)
                """,
                address_records
            )

        # Save this chunk
        conn.commit()

        total += len(chunk)

        # Free memory
        del records
        del prefix_records
        del first_records
        del address_records
        del chunk

        gc.collect()

        # Progress
        if total % 250_000 < CHUNK_SIZE:
            print(f"Processed {total:,} rows")

    print(f"\nFinished {source_name}: {total:,} rows")

In [14]:
process_source_file(
    os.path.join(DATA_DIR, "train_source2.tsv"),
    "S2"
)


Processing S2
File: /home/sagemaker-user/Igniters_submission_file/train_source2.tsv
Processed 250,000 rows
Processed 500,000 rows
Processed 750,000 rows
Processed 1,000,000 rows
Processed 1,250,000 rows
Processed 1,500,000 rows
Processed 1,750,000 rows
Processed 2,000,000 rows
Processed 2,250,000 rows
Processed 2,500,000 rows
Processed 2,750,000 rows
Processed 3,000,000 rows
Processed 3,250,000 rows
Processed 3,500,000 rows
Processed 3,750,000 rows
Processed 4,000,000 rows
Processed 4,250,000 rows
Processed 4,500,000 rows
Processed 4,750,000 rows
Processed 5,000,000 rows

Finished S2: 5,034,616 rows


In [15]:
print("\nDatabase verification after S2")

tables = [
    "entities",
    "name_prefix_index",
    "first_token_index",
    "address_token_index"
]

for table in tables:
    count = conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table}: {count:,}")


Database verification after S2
entities: 5,034,616
name_prefix_index: 9,248,862
first_token_index: 8,618,084
address_token_index: 24,461,807


In [16]:
duplicate_count = conn.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT entity_id
        FROM entities
        GROUP BY entity_id
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

print("Duplicate entity IDs:", duplicate_count)

Duplicate entity IDs: 0


In [22]:
process_source_file(
    os.path.join(DATA_DIR, "train_source3.tsv"),
    "S3"
)


Processing S3
File: /home/sagemaker-user/Igniters_submission_file/train_source3.tsv
Processed 250,000 rows
Processed 500,000 rows
Processed 750,000 rows
Processed 1,000,000 rows
Processed 1,250,000 rows
Processed 1,500,000 rows
Processed 1,750,000 rows
Processed 2,000,000 rows
Processed 2,250,000 rows
Processed 2,500,000 rows
Processed 2,750,000 rows
Processed 3,000,000 rows
Processed 3,250,000 rows
Processed 3,500,000 rows
Processed 3,750,000 rows
Processed 4,000,000 rows
Processed 4,250,000 rows
Processed 4,500,000 rows
Processed 4,750,000 rows
Processed 5,000,000 rows
Processed 5,250,000 rows

Finished S3: 5,285,603 rows


In [23]:
# Cell: Final database verification after S3

print("\n" + "=" * 50)
print("DATABASE VERIFICATION AFTER S3")
print("=" * 50)

tables = [
    "entities",
    "name_prefix_index",
    "first_token_index",
    "address_token_index"
]

for table in tables:
    count = conn.execute(
        f"SELECT COUNT(*) FROM {table}"
    ).fetchone()[0]

    print(f"{table}: {count:,}")

print("\nExpected entity count:")
print("S2:    5,034,616")
print("S3:    5,285,603")
print("Total: 10,320,219")


DATABASE VERIFICATION AFTER S3
entities: 10,320,219
name_prefix_index: 14,290,822
first_token_index: 13,321,683
address_token_index: 38,165,217

Expected entity count:
S2:    5,034,616
S3:    5,285,603
Total: 10,320,219


In [24]:
# Cell: Check Source 2 and Source 3 distribution

print("\n" + "=" * 50)
print("SOURCE DISTRIBUTION")
print("=" * 50)

source_counts = pd.read_sql_query("""
    SELECT source, COUNT(*) AS count
    FROM entities
    GROUP BY source
    ORDER BY source
""", conn)

print(source_counts.to_string(index=False))

print("\nExpected:")
print("S2:", f"{5_034_616:,}")
print("S3:", f"{5_285_603:,}")
print("Total:", f"{10_320_219:,}")


SOURCE DISTRIBUTION
source   count
    S2 5034616
    S3 5285603

Expected:
S2: 5,034,616
S3: 5,285,603
Total: 10,320,219


In [25]:
# Cell: Load a representative S1 sample and ground truth

S1_SAMPLE_SIZE = 20_000

print("Loading S1 sample...")

s1_sample = pd.read_csv(
    os.path.join(DATA_DIR, "train_source1.tsv"),
    sep="\t",
    dtype="string",
    nrows=S1_SAMPLE_SIZE
).fillna("")

ground_truth_sample = pd.read_csv(
    os.path.join(DATA_DIR, "train_ground_truth.tsv"),
    sep="\t",
    dtype="string",
    nrows=S1_SAMPLE_SIZE
).fillna("")

print("\nS1 sample shape:", s1_sample.shape)
print("Ground truth sample shape:", ground_truth_sample.shape)

print("\nS1 columns:")
print(s1_sample.columns.tolist())

print("\nGround truth columns:")
print(ground_truth_sample.columns.tolist())

print("\nFirst 3 S1 records:")
print(s1_sample.head(3).to_string(index=False))

print("\nFirst 3 ground truth records:")
print(ground_truth_sample.head(3).to_string(index=False))

Loading S1 sample...

S1 sample shape: (20000, 4)
Ground truth sample shape: (20000, 2)

S1 columns:
['entity_id', 'business_name', 'business_address', 'country']

Ground truth columns:
['source1_entity_id', 'matched_entity_ids']

First 3 S1 records:
   entity_id       business_name                       business_address country
S1-925783039 Orelee's Barbershop 1795 Westchester Drive, High Point, NC      US
S1-773889195         Prime Money        17560 Ellis Road, Tahlequah, OK      US
S1-377745466       B+ Retail Inc    1712 Montebello Avenue, Phoenix, AZ      US

First 3 ground truth records:
source1_entity_id                                              matched_entity_ids
        S1-965667 S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364
      S1-55344266             S2-249013014,S2-197070651,S3-478195123,S3-384364074
     S1-343815751                          S2-790675320,S2-479876582,S3-878454467


In [26]:
# Cell: Align ground truth with the S1 sample

# Create a lookup from Source 1 ID -> ground-truth matches
gt_lookup = dict(
    zip(
        ground_truth_sample["source1_entity_id"],
        ground_truth_sample["matched_entity_ids"]
    )
)

# Keep only S1 records whose IDs exist in the sampled ground truth
s1_sample["matched_entity_ids"] = (
    s1_sample["entity_id"]
    .map(gt_lookup)
    .fillna("")
)

print("S1 sample shape:", s1_sample.shape)

print("\nS1 records with ground truth available:",
      (s1_sample["matched_entity_ids"] != "").sum())

print("S1 records with no match:",
      (s1_sample["matched_entity_ids"] == "").sum())

print("\nSample after alignment:")
print(
    s1_sample[
        ["entity_id", "business_name", "country", "matched_entity_ids"]
    ].head(10).to_string(index=False)
)

S1 sample shape: (20000, 5)

S1 records with ground truth available: 174
S1 records with no match: 19826

Sample after alignment:
   entity_id                            business_name country matched_entity_ids
S1-925783039                      Orelee's Barbershop      US                   
S1-773889195                              Prime Money      US                   
S1-377745466                            B+ Retail Inc      US                   
S1-133037285                            Christ Chapel      US                   
S1-755362802                  Prabhav Business Center   India                   
S1-851869949               Custom Wealth Services LLC      US                   
S1-785847572 Consulting Nyasa Nursing Private Limited   India                   
 S1-27541239                        Nexus Anchor Rain      US                   
S1-629417405                        Moore Bitwise Inc      US                   
 S1-22305073               Dermatology Green Medicine      U

In [27]:
# Cell: Create correctly aligned evaluation sample

EVAL_SIZE = 20_000

print("Loading ground truth...")

gt_all = pd.read_csv(
    os.path.join(DATA_DIR, "train_ground_truth.tsv"),
    sep="\t",
    dtype="string"
).fillna("")

print("Ground truth rows:", f"{len(gt_all):,}")

# Randomly select ground-truth records
gt_eval = gt_all.sample(
    n=EVAL_SIZE,
    random_state=RANDOM_SEED
).copy()

# IDs we need from Source 1
eval_ids = set(gt_eval["source1_entity_id"])

print("Selected evaluation IDs:", f"{len(eval_ids):,}")

del gt_all
gc.collect()

# Now read Source 1 in chunks and retrieve only those IDs
s1_eval_parts = []

for chunk in pd.read_csv(
    os.path.join(DATA_DIR, "train_source1.tsv"),
    sep="\t",
    dtype="string",
    chunksize=25_000
):
    matches = chunk[chunk["entity_id"].isin(eval_ids)]

    if len(matches) > 0:
        s1_eval_parts.append(matches)

    if sum(len(x) for x in s1_eval_parts) >= EVAL_SIZE:
        break

s1_eval = pd.concat(
    s1_eval_parts,
    ignore_index=True
).fillna("")

# Attach ground truth by entity ID
gt_lookup = dict(
    zip(
        gt_eval["source1_entity_id"],
        gt_eval["matched_entity_ids"]
    )
)

s1_eval["matched_entity_ids"] = (
    s1_eval["entity_id"]
    .map(gt_lookup)
    .fillna("")
)

print("\n" + "=" * 50)
print("EVALUATION SAMPLE")
print("=" * 50)

print("S1 evaluation rows:", f"{len(s1_eval):,}")
print(
    "Rows with ground truth:",
    f"{(s1_eval['entity_id'].isin(eval_ids)).sum():,}"
)

print(
    "Rows with at least one match:",
    f"{(s1_eval['matched_entity_ids'] != '').sum():,}"
)

print(
    "Rows with zero matches:",
    f"{(s1_eval['matched_entity_ids'] == '').sum():,}"
)

print("\nFirst 10 records:")
print(
    s1_eval[
        ["entity_id", "business_name", "country", "matched_entity_ids"]
    ].head(10).to_string(index=False)
)

Loading ground truth...
Ground truth rows: 2,206,821
Selected evaluation IDs: 20,000

EVALUATION SAMPLE
S1 evaluation rows: 20,000
Rows with ground truth: 20,000
Rows with at least one match: 18,881
Rows with zero matches: 1,119

First 10 records:
   entity_id                   business_name country                                                                                                           matched_entity_ids
S1-708796012       Board of Consumer Affairs      US                                                S2-994438467,S2-440628035,S3-338279058,S3-335174066,S3-703951662,S3-974934575
S1-387694500    Bombay Power Private Limited   India S2-89272121,S2-828925482,S2-960848454,S2-674345300,S3-789542813,S3-9126403,S3-469722658,S3-904486811,S3-77197957,S3-91755708
S1-973233920         Data Materials Networks      US                                                                          S2-428307002,S3-110090880,S3-159133847,S3-792460305
S1-769172378 Technocast Chit Private Lim

In [ ]:
# Cell: Candidate generation functions

def get_candidates_for_s1(row):
    """
    Retrieve potential S2/S3 matches using the SQLite blocking indexes.
    """

    entity_id = str(row["entity_id"])
    country = normalize_text(row["country"])
    name = normalize_text(row["business_name"])
    address = normalize_text(row["business_address"])

    candidate_ids = set()

    # -------------------------------------------------
    # 1. Name prefix block
    # -------------------------------------------------
    if country and name:
        prefix = name[:4]

        if prefix:
            rows = conn.execute(
                """
                SELECT entity_id
                FROM name_prefix_index
                WHERE block_key = ?
                """,
                (country + "|" + prefix,)
            ).fetchall()

            candidate_ids.update(r[0] for r in rows)

    # -------------------------------------------------
    # 2. First name token block
    # -------------------------------------------------
    if country and name:
        name_parts = name.split()

        if name_parts:
            first = name_parts[0]

            if len(first) >= 3:
                rows = conn.execute(
                    """
                    SELECT entity_id
                    FROM first_token_index
                    WHERE block_key = ?
                    """,
                    (country + "|" + first,)
                ).fetchall()

                candidate_ids.update(r[0] for r in rows)

    # -------------------------------------------------
    # 3. Address token blocks
    # -------------------------------------------------
    if country and address:

        tokens = address.split()
        used = set()

        for token in tokens:

            if len(token) < 5:
                continue

            if token.isdigit():
                continue

            if token in used:
                continue

            used.add(token)

            rows = conn.execute(
                """
                SELECT entity_id
                FROM address_token_index
                WHERE block_key = ?
                """,
                (country + "|" + token,)
            ).fetchall()

            candidate_ids.update(r[0] for r in rows)

            if len(used) >= 3:
                break

    # Don't return the S1 ID itself if somehow present
    candidate_ids.discard(entity_id)

    return candidate_ids


print("Candidate generation function ready.")

In [33]:
# FAST candidate generation test
# Test only ONE S1 record

row = s1_eval.iloc[0]

print("Testing:", row["entity_id"])
print("Business:", row["business_name"])
print("Country:", row["country"])

candidates = get_candidates_for_s1(row)

print("\nNumber of candidates:", len(candidates))
print("Sample candidates:", list(candidates)[:10])
print("\nDone!")

Testing: S1-708796012
Business: Board of Consumer Affairs
Country: US

Number of candidates: 53869
Sample candidates: ['S2-792816639', 'S3-428240212', 'S2-738061232', 'S3-325916292', 'S3-747842693', 'S2-331307389', 'S2-507798160', 'S3-399969514', 'S2-933687053', 'S3-410602464']

Done!


In [34]:
row = s1_eval.iloc[0]

country = normalize_text(row["country"])
name = normalize_text(row["business_name"])
address = normalize_text(row["business_address"])

print("Country:", country)
print("Name:", name)
print("Address:", address)

# -----------------------------------
# 1. NAME PREFIX
# -----------------------------------

prefix = name[:4]

prefix_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM name_prefix_index
    WHERE block_key = ?
    """,
    (country + "|" + prefix,)
).fetchone()[0]

print("\nName prefix:", prefix)
print("Name-prefix candidates:", prefix_count)


# -----------------------------------
# 2. FIRST NAME TOKEN
# -----------------------------------

name_parts = name.split()

if name_parts:
    first = name_parts[0]

    first_count = conn.execute(
        """
        SELECT COUNT(*)
        FROM first_token_index
        WHERE block_key = ?
        """,
        (country + "|" + first,)
    ).fetchone()[0]

    print("\nFirst name token:", first)
    print("First-token candidates:", first_count)


# -----------------------------------
# 3. ADDRESS TOKENS
# -----------------------------------

print("\nAddress token candidates:")

for token in address.split():

    if len(token) < 5:
        continue

    if token.isdigit():
        continue

    count = conn.execute(
        """
        SELECT COUNT(*)
        FROM address_token_index
        WHERE block_key = ?
        """,
        (country + "|" + token,)
    ).fetchone()[0]

    print(f"{token}: {count}")

Country: us
Name: board of consumer affairs
Address: 122 dayton yellow springs road unit 4 fairborn oh

Name prefix: boar
Name-prefix candidates: 10215

First name token: board
First-token candidates: 9598

Address token candidates:
dayton: 18940
yellow: 2195
springs: 50283
fairborn: 2075


In [35]:
def get_candidates_for_s1(row, max_candidates=3000):
    """
    Generate controlled candidates using multiple blocking signals.

    Strategy:
    1. Collect candidates from name and address blocks.
    2. Count how many independent blocks each candidate matches.
    3. Prefer candidates matching multiple blocks.
    4. Keep at most max_candidates.
    """

    entity_id = str(row["entity_id"])
    country = normalize_text(row["country"])
    name = normalize_text(row["business_name"])
    address = normalize_text(row["business_address"])

    candidate_scores = {}

    def add_candidates(rows):
        for (candidate_id,) in rows:
            if candidate_id == entity_id:
                continue

            candidate_scores[candidate_id] = (
                candidate_scores.get(candidate_id, 0) + 1
            )

    # ------------------------------------------------
    # 1. Name prefix block
    # ------------------------------------------------

    if country and name:

        prefix = name[:4]

        if prefix:

            rows = conn.execute(
                """
                SELECT entity_id
                FROM name_prefix_index
                WHERE block_key = ?
                """,
                (country + "|" + prefix,)
            ).fetchall()

            add_candidates(rows)

    # ------------------------------------------------
    # 2. First name token block
    # ------------------------------------------------

    if country and name:

        name_parts = name.split()

        if name_parts:

            first = name_parts[0]

            if len(first) >= 3:

                rows = conn.execute(
                    """
                    SELECT entity_id
                    FROM first_token_index
                    WHERE block_key = ?
                    """,
                    (country + "|" + first,)
                ).fetchall()

                add_candidates(rows)

    # ------------------------------------------------
    # 3. Address blocks
    # ------------------------------------------------

    if country and address:

        tokens = address.split()

        used = set()

        for token in tokens:

            if len(token) < 5:
                continue

            if token.isdigit():
                continue

            if token in used:
                continue

            used.add(token)

            rows = conn.execute(
                """
                SELECT entity_id
                FROM address_token_index
                WHERE block_key = ?
                """,
                (country + "|" + token,)
            ).fetchall()

            add_candidates(rows)

            if len(used) >= 3:
                break

    # ------------------------------------------------
    # Rank candidates by number of matching blocks
    # ------------------------------------------------

    ranked_candidates = sorted(
        candidate_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Keep only the strongest candidates
    ranked_candidates = ranked_candidates[:max_candidates]

    return set(
        candidate_id
        for candidate_id, score in ranked_candidates
    )


print("Updated candidate-generation function.")

Updated candidate-generation function.


In [37]:
# Test the updated candidate generator

row = s1_eval.iloc[0]

print("Testing:", row["entity_id"])
print("Business:", row["business_name"])
print("Country:", row["country"])

candidates = get_candidates_for_s1(row)

print("\nNumber of candidates:", len(candidates))
print("Sample candidates:", list(candidates)[:10])

Testing: S1-708796012
Business: Board of Consumer Affairs
Country: US

Number of candidates: 3000
Sample candidates: ['S2-45166195', 'S2-738061232', 'S2-467456381', 'S2-455976775', 'S2-249952177', 'S2-738400833', 'S2-561886678', 'S2-609797595', 'S2-908426961', 'S2-87373430']


In [38]:
# Check whether the real matches are still inside our 3,000 candidates

row = s1_eval.iloc[0]

candidates = get_candidates_for_s1(row)

# Ground-truth matches
true_matches = set()

if row["matched_entity_ids"]:
    true_matches = set(
        str(row["matched_entity_ids"]).split(",")
    )

found_matches = true_matches.intersection(candidates)
missing_matches = true_matches - candidates

print("=" * 50)
print("GROUND TRUTH CHECK")
print("=" * 50)

print("S1:", row["entity_id"])
print("Business:", row["business_name"])

print("\nTrue matches:", len(true_matches))
print("Found in candidates:", len(found_matches))
print("Missing:", len(missing_matches))

print("\nTrue matches:")
print(sorted(true_matches))

print("\nFound matches:")
print(sorted(found_matches))

print("\nMissing matches:")
print(sorted(missing_matches))

if len(true_matches) > 0:
    recall = len(found_matches) / len(true_matches)
    print("\nCandidate recall:", f"{recall:.2%}")

GROUND TRUTH CHECK
S1: S1-708796012
Business: Board of Consumer Affairs

True matches: 6
Found in candidates: 5
Missing: 1

True matches:
['S2-440628035', 'S2-994438467', 'S3-335174066', 'S3-338279058', 'S3-703951662', 'S3-974934575']

Found matches:
['S2-440628035', 'S2-994438467', 'S3-335174066', 'S3-338279058', 'S3-974934575']

Missing matches:
['S3-703951662']

Candidate recall: 83.33%


In [42]:
# Test a larger candidate limit

row = s1_eval.iloc[0]

candidates = get_candidates_for_s1(
    row,
    max_candidates=10000
)

true_matches = set(
    str(row["matched_entity_ids"]).split(",")
)

found_matches = true_matches.intersection(candidates)
missing_matches = true_matches - candidates

print("=" * 50)
print("10,000 CANDIDATE TEST")
print("=" * 50)

print("Total candidates:", len(candidates))
print("True matches:", len(true_matches))
print("Found:", len(found_matches))
print("Missing:", len(missing_matches))

print("\nMissing matches:")
print(sorted(missing_matches))

recall = len(found_matches) / len(true_matches)

print("\nCandidate recall:", f"{recall:.2%}")

10,000 CANDIDATE TEST
Total candidates: 10000
True matches: 6
Found: 6
Missing: 0

Missing matches:
[]

Candidate recall: 100.00%


In [43]:
# Candidate recall test on 100 evaluation records

TEST_SAMPLE_SIZE = 100
MAX_CANDIDATES = 10000

test_subset = s1_eval.head(TEST_SAMPLE_SIZE).copy()

candidate_counts = []
true_match_counts = []
found_true_match_counts = []

for i, (_, row) in enumerate(test_subset.iterrows(), start=1):

    candidates = get_candidates_for_s1(
        row,
        max_candidates=MAX_CANDIDATES
    )

    true_matches = set()

    if row["matched_entity_ids"]:
        true_matches = set(
            str(row["matched_entity_ids"]).split(",")
        )

    found_true = true_matches.intersection(candidates)

    candidate_counts.append(len(candidates))
    true_match_counts.append(len(true_matches))
    found_true_match_counts.append(len(found_true))

    if i % 10 == 0:
        print(f"Processed {i}/{TEST_SAMPLE_SIZE}")


total_true = sum(true_match_counts)
total_found = sum(found_true_match_counts)

recall = (
    total_found / total_true
    if total_true > 0
    else 0
)

print("\n" + "=" * 50)
print("100-RECORD CANDIDATE RECALL TEST")
print("=" * 50)

print("S1 records tested:", TEST_SAMPLE_SIZE)

print(
    "Average candidates:",
    round(np.mean(candidate_counts), 2)
)

print(
    "Minimum candidates:",
    min(candidate_counts)
)

print(
    "Maximum candidates:",
    max(candidate_counts)
)

print(
    "Average true matches:",
    round(np.mean(true_match_counts), 2)
)

print(
    "True matches found:",
    f"{total_found:,} / {total_true:,}"
)

print(
    "Candidate recall:",
    f"{recall:.2%}"
)

Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Processed 100/100

100-RECORD CANDIDATE RECALL TEST
S1 records tested: 100
Average candidates: 9331.76
Minimum candidates: 945
Maximum candidates: 10000
Average true matches: 3.65
True matches found: 320 / 365
Candidate recall: 87.67%
